# Message checks

> Check a message against its triage record without labels: format, coverage, severity, numbers and length

In [ ]:
#| default_exp checks

The [scope](design/scope.ipynb) lists failure modes that code can check by comparing a message with its triage record, with no labels needed. This module implements them for the first task. The same checks screen the teacher's messages before they become training data, and score the base and fine-tuned models, broken down by tuple.

In [ ]:
#| export
import re

In [ ]:
#| hide
import random
from fastcore.test import *
from edgeslm.coffee import *

## A record and its message

The examples use one record from the generator in [`coffee`](01_coffee.ipynb), with a severe, a moderate and an early item:

In [ ]:
rec = triage(dict(problem='both', pattern='cluster', severity='severe'), random.Random(25))
rec

{'farm': 'Sítio Recanto',
 'date': '2026-09-09',
 'nodes': 17,
 'items': [{'problem': 'equipment fault',
   'location': 'Block C, drip line C3',
   'pattern': 'cluster',
   'severity': 'severe',
   'since': 'for a week or more',
   'likely_cause': 'no data from these nodes: radio link lost',
   'nodes_flagged': 2,
   'nodes_in_group': 8},
  {'problem': 'water stress',
   'location': 'Block B, drip line B1',
   'pattern': 'cluster',
   'severity': 'moderate',
   'since': 'since Monday',
   'likely_cause': 'low water pressure',
   'nodes_flagged': 2,
   'nodes_in_group': 4},
  {'problem': 'equipment fault',
   'location': 'Block A, row 28',
   'pattern': 'one node',
   'severity': 'early',
   'since': 'since today',
   'likely_cause': 'soil sensor gives implausible values'}]}

A message that keeps every rule has one paragraph per item. Each paragraph starts with the item's marker and its location exactly as in the record, then a colon, then what is wrong and what to do:

In [ ]:
msg = """🚨 Block C, drip line C3: these sensors have sent no data for a week or more, so the plants on this line are unmonitored. Check the radio link of the nodes.

⚠️ Block B, drip line B1: some monitored plants have been short of water since Monday. Check the water pressure on the line.

👀 Block A, row 28: the soil sensor gives values that look wrong since today. Check that it sits firmly in the soil."""

## Parsing

`parse` splits a message into paragraphs, and reads the marker, location and text of each. Models sometimes drop the invisible variation selector that follows ⚠ in ⚠️, so `parse` accepts the marker with or without it.

In [ ]:
#| export
MARKERS = dict(early='👀', moderate='⚠️', severe='🚨')
_item_re = re.compile(r'(👀|⚠️|🚨)\s*([^:\n]+?)\s*:\s*(.+)', re.S)

def parse(
    msg:str, # A message, with one paragraph per item
)->list: # (marker, location, text) for each paragraph, or `None` for one not in that format
    "Split `msg` into its items"
    msg = re.sub('\N{WARNING SIGN}\N{VARIATION SELECTOR-16}?', '\N{WARNING SIGN}\N{VARIATION SELECTOR-16}', msg)
    paras = [p.strip() for p in re.split(r'\n\s*\n', msg.strip()) if p.strip()]
    return [m.groups() if (m := _item_re.fullmatch(p)) else None for p in paras]

In [ ]:
ps = parse(msg)
test_eq([(m, l) for m, l, _ in ps], [('🚨', 'Block C, drip line C3'), ('⚠️', 'Block B, drip line B1'), ('👀', 'Block A, row 28')])
ps[0]

('🚨',
 'Block C, drip line C3',
 'these sensors have sent no data for a week or more, so the plants on this line are unmonitored. Check the radio link of the nodes.')

A paragraph without a marker and a location gives `None`:

In [ ]:
test_eq(parse("Check your farm today.\n\n⚠ Block B, drip line B1: check the line."), [None, ('⚠️', 'Block B, drip line B1', 'check the line.')])

## Checks

`check` runs the checks of the scope, and returns whether each passed:

- `format`: every paragraph has a marker and a location.
- `coverage`: every item of the record has its paragraph.
- `no_invented`: every paragraph names a location from the record, and no location has two paragraphs.
- `severity`: each paragraph has the marker of its item's severity.
- `no_numbers`: no digits outside the record's locations, so no readings, counts or dates.
- `length`: at most `MAX_LEN` characters.

Locations are compared without case, so that "Whole farm" at the start of a paragraph matches `whole farm`.

In [ ]:
#| export
MAX_LEN = 600

def check(
    rec:dict, # Triage record
    msg:str, # Message for `rec`
)->dict: # Check name -> passed
    "Run the code checks on `msg`"
    items = {it['location'].casefold(): it for it in rec['items']}
    ps = parse(msg)
    found = [(m, l.casefold()) for m, l, _ in filter(None, ps)]
    locs = [l for _, l in found]
    rest = re.sub('|'.join(re.escape(l) for l in sorted(items, key=len, reverse=True)), '', msg, flags=re.I)
    return dict(
        format=bool(ps) and all(ps), coverage=set(items) <= set(locs),
        no_invented=set(locs) <= set(items) and len(locs) == len(set(locs)),
        severity=all(m == MARKERS[items[l]['severity']] for m, l in found if l in items),
        no_numbers=not re.search(r'\d', rest), length=len(msg) <= MAX_LEN)

The message above passes every check:

In [ ]:
res = check(rec, msg)
assert all(res.values())
res

{'format': True,
 'coverage': True,
 'no_invented': True,
 'severity': True,
 'no_numbers': True,
 'length': True}

Each check catches one kind of mistake. This message gives a date, marks the second item as severe, and leaves out the third item, so it fails three checks:

In [ ]:
bad = """🚨 Block C, drip line C3: these sensors have sent no data since 1 September. Check the radio link of the nodes.

🚨 Block B, drip line B1: some monitored plants have been short of water since Monday. Check the water pressure on the line."""
res = check(rec, bad)
test_eq({k for k, v in res.items() if not v}, {'no_numbers', 'severity', 'coverage'})
res

{'format': True,
 'coverage': False,
 'no_invented': True,
 'severity': False,
 'no_numbers': False,
 'length': True}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()